In [1]:
import json
from sentence_transformers import SentenceTransformer, util

# model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

data = []
count_event_exists = 0
file_path = "/home/mhtuan/work/mbf/result/filtered_matched_date.json"
with open(file_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        json_rec = json.loads(line)
        data.append(json_rec)

/home/mhtuan/anaconda3/envs/ne-env-1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
set_ignore_trigger_type = {
    'Arquit',
    'Divorce',
    "Declare-bankruptcy",
    "Pardon",
    "Release-parole",
    "Demonstrate",
    "Execute",
    "Fine",
    "Injure",
    "Start-position",
    "Start-org",
    "Nominate",
    "Elect",
    "Meet",
    "Sue",
    "Be-born",
    "Attack",
    "Convict",
    "Phone-write",
    "Trial-hearing",
    "Transport",
}

In [2]:
import math
from collections import Counter
import re
from functools import lru_cache

RE_HASHTAG = re.compile("#\w*", flags=re.UNICODE)
RE_URL = r'https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&\/\/=]*)'
RE_SPECIAL_CHARS = r'[\.\,\!\@\#\$\%\^\&\*\(\)\=\|\"\'\;\:\‘\’\“\”\‼️\-\[\]\{\}\?\<\>\…\/\\\~\+]'

def strip_hashtag(text: str) -> str:
    return RE_HASHTAG.sub(r'', text)

def strip_url(text: str) -> str:
    return re.sub(RE_URL, '', text)

def strip_special_chars(text: str) -> str:
    return re.sub(RE_SPECIAL_CHARS, '', text)

def clean_text(text: str) -> str:
    processed_text = text.strip().replace("\n", " ").replace("\t", " ")
    processed_text = strip_hashtag(processed_text)
    processed_text = strip_url(processed_text)
    processed_text = strip_special_chars(processed_text)
    return processed_text

@lru_cache(maxsize=10000)
def get_cosine(text1: str, text2: str):
    vec1 = Counter(text1.split())
    vec2 = Counter(text2.split())

    intersection = set(vec1.keys()) & set(vec2.keys())
    numerator = sum(vec1[x] * vec2[x] for x in intersection)

    sum1 = sum(vec1[x] ** 2 for x in vec1)
    sum2 = sum(vec2[x] ** 2 for x in vec2)
    denominator = math.sqrt(sum1) * math.sqrt(sum2)

    return 0.0 if denominator == 0 else numerator / denominator
    
def is_abbrev(abbrev, text):
    abbrev=abbrev.lower()
    text=text.lower()
    words=text.split()
    if not abbrev:
        return True
    if abbrev and not text:
        return False
    if abbrev[0]!=text[0]:
        return False
    else:
        return (is_abbrev(abbrev[1:],' '.join(words[1:])) or
                any(is_abbrev(abbrev[1:],text[i+1:])
                    for i in range(len(words[0]))))
    
def string_is_similar(sentence1, sentence2, use_abbrev=True, model=None, score=0.7):
    cleaned_text_1 = clean_text(sentence1)
    cleaned_text_2 = clean_text(sentence2)

    if cleaned_text_1 == cleaned_text_2:
        return True
    
    if not use_abbrev:
        return False
    
    if (len(cleaned_text_1) < len(cleaned_text_2) and is_abbrev(abbrev=cleaned_text_1, text=cleaned_text_2)) or\
        (len(cleaned_text_1) > len(cleaned_text_2) and is_abbrev(abbrev=cleaned_text_2, text=cleaned_text_1)):
        return True
    
    return False
    # if get_cosine(cleaned_text_1, cleaned_text_2) >= 0.6:
    #     return True

    # return False
    # embedding1 = model.encode(sentence1, convert_to_tensor=True)
    # embedding2 = model.encode(sentence2, convert_to_tensor=True)

    # similarity = util.pytorch_cos_sim(embedding1, embedding2).item()

    # print(similarity)
    # return similarity >= score

sentence1 = "học sinh trường phổ thông dân tộc bán trú tiểu học trà dơn , huyện nam trà my , tỉnh quảng nam"
sentence2 = "hai học sinh lớp 9"
print(string_is_similar(sentence1, sentence2))

False


In [4]:
import json

def extract_schema(record: dict):
    triggers_dict = dict()
    grouping_by_event_dict = dict()

    url = record.get('url')
    date = record.get('date')
    sentence = record.get('text')

    # if len(set([e['text'] for e in record['entity_mentions']])) < 2:
    #     return None, url, date, sentence
    
    for trigger in record['event_triggers']:
        # if trigger['event_type'] in set_ignore_trigger_type:
        #     continue

        triggers_dict[trigger['trigger_id']] = [trigger['event_type'], trigger['text'].lower()]
    
    for arg in record['event_arguments']:
        trigger_id = arg['trigger_id']
        if trigger_id not in triggers_dict:
            continue
        
        entity_id = arg['entity_id']
        role_type = arg['role_type']
        if trigger_id not in grouping_by_event_dict:
            grouping_by_event_dict[trigger_id] = {}

        for entity in record['entity_mentions']:
            if entity['entity_id'] == entity_id:
                constructed_entity = {"role_type": role_type, "entity_type": entity['entity_type'], "text": entity['text'].lower()}
                constructed_schema = { "text": triggers_dict[trigger_id][1], "entities": [constructed_entity] }
                
                if triggers_dict[trigger_id][0] not in grouping_by_event_dict[trigger_id]:
                    grouping_by_event_dict[trigger_id].update({
                        triggers_dict[trigger_id][0]: constructed_schema
                    })
                else:
                    # print(grouping_by_event_dict[triggers_dict[trigger_id][0]])
                    grouping_by_event_dict[trigger_id][triggers_dict[trigger_id][0]]['entities'].append(constructed_entity)

                break

    return list(grouping_by_event_dict.values()), url, date, sentence



In [ ]:
dict_by_event = dict()
for item in data:
    schema, url, date, sentence = extract_schema(item)
    for sch in schema:
        for trigger in sch:
            trigger_text = sch[trigger]['text']
            entities = {"entities": sch[trigger]['entities'], "url": url, "date": date, "sentence": sentence}

            if trigger not in dict_by_event:
                dict_by_event[trigger] = {trigger_text: [entities]}
            else:
                if trigger_text not in dict_by_event[trigger]:
                    dict_by_event[trigger].update({trigger_text: [entities]})
                else:
                    dict_by_event[trigger][trigger_text].append(entities)

with open("grouping.json", "w+") as f:
    f.write(json.dumps(dict_by_event, ensure_ascii=False))

In [5]:
clustering_groups = dict()

def brute_force_two_sets(current: set, avai: set, use_abbrev=True, score=0.7, divide_coefficient=2):
    count_similar = 0

    for c in current:
        for a in avai:
            if string_is_similar(c, a, use_abbrev=use_abbrev, score=score):
                count_similar += 1
            
    return count_similar >= (len(current) / divide_coefficient)

def compare_two_triggers_sets(current: set, avai: set) -> bool:
    if len(current.intersection(avai)) > 0:
        return True
    return brute_force_two_sets(current, avai, use_abbrev=False, score=0.5, divide_coefficient=1)

def compare_two_entities_sets(current: set, avai: set) -> bool:
    intersection = current.intersection(avai)
    missing = current - avai

    if len(intersection) == len(current):
        return True
    elif len(intersection) > 0:
        return brute_force_two_sets(missing, avai, score=0.6)
    else:
        return brute_force_two_sets(current, avai, divide_coefficient=1)

def find_best_group(group: dict, clustering_groups: list):
    current_group_triggers = group['triggers']
    current_group_entities = set()
    for ent in group['entities']:
        current_group_entities.update(ent['texts'])

    group_founded = False
    for avai_group in clustering_groups:
        avai_group_triggers = avai_group['triggers']
        avai_group_entities = set()
        for ent in avai_group['entities']:
            avai_group_entities.update(ent['texts'])

        if not compare_two_triggers_sets(current={current_group_triggers}, avai={avai_group_triggers}):
            continue

        avai_group['entities'] += group['entities']
        group_founded = True
        # if compare_two_entities_sets(current=current_group_entities, avai=avai_group_entities):
        #     avai_group['entities'] += group['entities']
        #     # avai_group['triggers'].update(current_group_triggers)
        #     group_founded = True
    
    if not group_founded:
        clustering_groups.append(group)

for index, item in enumerate(data):
    print("event", index)
    schema, url, date, sentence = extract_schema(item)
    if schema == None:
        continue
    for sch in schema:
        for trigger in sch:
            if trigger not in clustering_groups:
                clustering_groups[trigger] = []

            trigger_text = sch[trigger]['text']

            entities_text_set = set([e['text'] for e in sch[trigger]['entities']])
            # if len(entities_text_set) < 2:
            #     continue

            entities = [{"texts": entities_text_set, "url": url, "date": date, "sentence": sentence}]
            group = {"triggers": trigger_text, "trigger_type": trigger, "entities": entities}

            if len(clustering_groups[trigger]) == 0:
                clustering_groups[trigger].append(group)
            else:
                find_best_group(group=group, clustering_groups=clustering_groups[trigger])
            
        

event 0
event 1
event 2
event 3
event 4
event 5
event 6
event 7
event 8
event 9
event 10
event 11
event 12
event 13
event 14
event 15
event 16
event 17
event 18
event 19
event 20
event 21
event 22
event 23
event 24
event 25
event 26
event 27
event 28
event 29
event 30
event 31
event 32
event 33
event 34
event 35
event 36
event 37
event 38
event 39
event 40
event 41
event 42
event 43
event 44
event 45
event 46
event 47
event 48
event 49
event 50
event 51
event 52
event 53
event 54
event 55
event 56
event 57
event 58
event 59
event 60
event 61
event 62
event 63
event 64
event 65
event 66
event 67
event 68
event 69
event 70
event 71
event 72
event 73
event 74
event 75
event 76
event 77
event 78
event 79
event 80
event 81
event 82
event 83
event 84
event 85
event 86
event 87
event 88
event 89
event 90
event 91
event 92
event 93
event 94
event 95
event 96
event 97
event 98
event 99
event 100
event 101
event 102
event 103
event 104
event 105
event 106
event 107
event 108
event 109
event 110


In [6]:
# filtered = []
for trigger_type in clustering_groups:
    for g in clustering_groups[trigger_type]:
        # g['triggers'] = list(g['triggers'])
        # if len(g['entities']) >= 2:
        #     filtered.append(g)
        for ent in g['entities']:
            ent['texts'] = list(ent['texts'])

In [7]:
with open("clustering_groups_all_types.json", "w+") as f:
    f.write(json.dumps(clustering_groups, ensure_ascii=False))

In [28]:
import math

def entities_similar(e1: str, e2: str, score=0.75):
    cleaned_text_1 = clean_text(e1)
    cleaned_text_2 = clean_text(e2)

    if cleaned_text_1 == cleaned_text_2:
        return True
    
    if ((cleaned_text_1 in cleaned_text_2 and len(cleaned_text_1.split()) >= 2 and len(cleaned_text_2.split()) > len(cleaned_text_1.split())) or 
        (cleaned_text_2 in cleaned_text_1 and len(cleaned_text_2.split()) >= 2 and len(cleaned_text_1.split()) > len(cleaned_text_2.split()))):
        return True
    
    if get_cosine(cleaned_text_1, cleaned_text_2) >= score:
        return True
    
    return False

def entities_sets_similar(set1: set, set2: set) -> bool:
    count_to_match = math.ceil(max(len(set1), len(set2)) / 2)

    if len(set1.intersection(set2)) >= count_to_match:
        return True
    
    count_similar = 0
    for e1 in set1:
        for e2 in set2:
            if entities_similar(e1, e2):
                count_similar += 1
                # return True
    
    return count_similar >= count_to_match

In [13]:
s1 = "nhà thơ , nhạc sĩ nguyễn thụy kha"
s2 = "nhạc sĩ , nhà thơ nguyễn thụy kha"
entities_sets_similar({s1}, {s2})

True

In [7]:
from transformers import AutoModel, AutoTokenizer, DistilBertTokenizerFast, DistilBertModel
import torch
import torch.nn.functional as F

# Load PhoBERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
model = AutoModel.from_pretrained("vinai/phobert-base-v2")



/home/mhtuan/anaconda3/envs/ne-env-1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
from functools import lru_cache

@lru_cache(maxsize=10000)
def get_embedding(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # print(outputs)
    return outputs.last_hidden_state[:, 0, :]

def cosine_string_similar(s1: str, s2: str, score=0.75) -> bool:
    embedding1 = get_embedding(s1)
    embedding2 = get_embedding(s2)

    similarity = F.cosine_similarity(embedding1, embedding2).item()

    # print(similarity)
    if similarity >= score:
        return True
    return False

In [14]:
s1 = "\" Nhà thơ , nhạc sĩ Nguyễn Thụy Kha qua đời \""
s2 = "\" Nhạc sĩ , nhà thơ Nguyễn Thụy Kha qua đời \""
cosine_string_similar(s1, s2)

0.9884209632873535


True

In [16]:
with open("vietnamese_stopwords.txt", "r") as f:
    lines = f.readlines()
    stopwords = set()
    for l in lines:
        stopwords.add(l.strip())

In [32]:
import json

with open("/home/mhtuan/work/mbf/clustering_groups_remove_types.json") as f:
    full_groups = json.load(f)

transfer_ownership = full_groups["Die"]
transfer_ownership_fine_grained_groups = []

print(f"total {len(transfer_ownership)} groups\n####################\n")
for raw_index, group in enumerate(transfer_ownership):
    print(f"group {raw_index}")
    triggers = group["triggers"]
    trigger_type = group["trigger_type"]
    entities = group["entities"]

    temp_group = dict()

    print(f'total {len(entities)} entity')
    for ent_index, ent in enumerate(entities):
        # print(f'entity {ent_index}')
        raw_sentence = ent['sentence']
        current_entities = set(ent['texts'])

        if len(stopwords.intersection(set(ent['texts']))) > 0:
            continue

        group = {"triggers": triggers, "trigger_type": trigger_type, "sentences": {raw_sentence}, "entities": [ent]}
        if len(temp_group) == 0:
            temp_group.update({ent_index: group})
            continue

        group_found = False
        for avai_group in temp_group.values():
            if triggers != avai_group['triggers']:
                continue

            available_entities = set()
            for ae in avai_group['entities']:
                for t in ae['texts']:
                    available_entities.add(t)
            available_sentences = avai_group['sentences'].copy()

            if raw_sentence in available_sentences:
                avai_group['sentences'].add(raw_sentence)
                avai_group['entities'].append(ent)
                group_found = True
                break

            for avai_sentence in available_sentences:
                try:
                    if cosine_string_similar(raw_sentence, avai_sentence, score=0.825) and entities_sets_similar(current_entities, available_entities):
                        avai_group['sentences'].add(raw_sentence)
                        avai_group['entities'].append(ent)
                        group_found = True
                        break
                    # else:
                    #     print("Failed")
                    #     print(raw_sentence, "\n", avai_sentence)
                    #     print(current_entities, "\n", available_entities)
                    #     print("\n")
                except Exception as e:
                    continue

            if group_found:
                break

        if not group_found:
            temp_group.update({ent_index: group})

    temp_keys = list(temp_group.keys())
    for key in temp_keys:
        if len(temp_group[key]["entities"]) <= 2:
            temp_group.pop(key)

    transfer_ownership_fine_grained_groups += list(temp_group.values())

total 156 groups
####################

group 0
total 95 entity
group 1
total 401 entity
group 2
total 57 entity
group 3
total 1135 entity
group 4
total 203 entity
group 5
total 2 entity
group 6
total 3 entity
group 7
total 76 entity
group 8
total 1 entity
group 9
total 104 entity
group 10
total 3 entity
group 11
total 99 entity
group 12
total 414 entity
group 13
total 19 entity
group 14
total 6 entity
group 15
total 29 entity
group 16
total 1 entity
group 17
total 84 entity
group 18
total 1 entity
group 19
total 30 entity
group 20
total 2 entity
group 21
total 8 entity
group 22
total 20 entity
group 23
total 2 entity
group 24
total 16 entity
group 25
total 27 entity
group 26
total 1 entity
group 27
total 2 entity
group 28
total 7 entity
group 29
total 13 entity
group 30
total 13 entity
group 31
total 2 entity
group 32
total 7 entity
group 33
total 6 entity
group 34
total 1 entity
group 35
total 6 entity
group 36
total 1 entity
group 37
total 2 entity
group 38
total 1 entity
group 39
to

In [33]:
for item in transfer_ownership_fine_grained_groups:
    item.pop("sentences")

In [34]:
with open("fine_grained_test_die.json", "w+") as f:
    f.write(json.dumps(transfer_ownership_fine_grained_groups, ensure_ascii=False))

--- Low Counts Test ---
Trend Score: 1.17
Trend Confirmed: False
Poisson p-values: 0.26424111765711533

--- High Counts Test ---
Trend Score: 2.75
Trend Confirmed: True
